# Module 4 · Serving, monitoring and cloud
    **Outcomes:** (g) serve a model as a REST API with FastAPI inside Docker; (h) monitor it in production and map the workflow to Azure ML.

> **How to use this notebook.** Run cells top to bottom with **Shift + Enter**.
> Cells marked **✏️ TODO** are yours to complete. **🏁 Checkpoint** cells tell you when a module is done.
> Stuck? Ask your neighbour first, then the trainer. Solutions are released after the session.

## A · Start the API (in a terminal, not here)
Open a terminal **at the repo root** with the environment active, then pick one:

**With Docker** (the checkpoint):
```bash
cd api
docker build -t loan-api:1.0 .
docker run -d -p 8000:8000 --name loan-api loan-api:1.0
```
**Without Docker** (same API, runs in your Python environment):
```bash
cd api
uvicorn main:app --reload --port 8000
```
Port 8000 taken? Use `-p 8001:8000` (Docker) or `--port 8001`, and change `API` below.

In [ ]:
import httpx
import matplotlib.pyplot as plt

from course_utils import REPO_ROOT, check_status_codes, dashboard_series

API = "http://localhost:8000"
try:
    r = httpx.get(f"{API}/health", timeout=3)
    print("✅", r.status_code, r.json(), "· Swagger docs at", f"{API}/docs")
except httpx.HTTPError as exc:
    print(f"❌ Nothing answering at {API} ({type(exc).__name__}). Start the API in a terminal first (see above).")

In [ ]:
def call(payload):
    """POST one application to /v1/predict and show what comes back."""
    try:
        r = httpx.post(f"{API}/v1/predict", json=payload, timeout=5)
    except httpx.HTTPError as exc:
        print(f"❌ API not reachable ({type(exc).__name__})")
        return None
    body = r.json()
    print(r.status_code, body if r.status_code == 200 else body["detail"][0]["msg"])
    return r.status_code

call({"age": 35, "income": 72000, "credit_score": 710})

## B · Hands-on: make Pydantic say no
The API declares `age` 18–120, `income` > 0, `credit_score` 300–850, all required.
Bad input should be rejected with **422** before it ever reaches the model.

In [ ]:
valid = {"age": 35, "income": 72000, "credit_score": 710}

# ✏️ TODO: copy `valid`, then change the age to one the API must refuse
age_too_low = {}
# ✏️ TODO: copy `valid`, then leave out one required field
missing_field = {}

results = {"valid": call(valid), "age_too_low": call(age_too_low), "missing_field": call(missing_field)}
check_status_codes(results, age_too_low, missing_field)

🏁 **Checkpoint · Module 4:** the container (or uvicorn) answers, and bad input gets a 422.
Try the same calls from the interactive docs at http://localhost:8000/docs.

## C · The MLOps dashboard and SLOs (15 min)
Ten weeks of production metrics for the loan model (illustrative). Alert only when a **promise (SLO)** breaks.

In [ ]:
data = dashboard_series()

# ✏️ TODO: set the service level objectives from the slides
SLO = {
    "auc_min": ...,              # model: monthly AUC must stay at or above this
    "top_feature_psi_max": ...,  # data: PSI of the top features must stay below this
    "p99_latency_ms_max": ...,   # system: p99 latency in ms
    "error_rate_pct_max": ...,   # system: share of failed requests, in %
}

In [ ]:
rules = {
    "Model · AUC": ("auc", lambda v: v >= SLO["auc_min"]),
    "Data · top feature PSI": ("top_feature_psi", lambda v: v < SLO["top_feature_psi_max"]),
    "System · p99 latency": ("p99_latency_ms", lambda v: v < SLO["p99_latency_ms_max"]),
    "System · error rate": ("error_rate_pct", lambda v: v < SLO["error_rate_pct_max"]),
}
latest = data.iloc[-1]
print(f"Latest week ({latest.week}):")
for name, (col, within) in rules.items():
    broken = [w for w, v in zip(data.week, data[col]) if not within(v)]
    state = "🟢 dashboard only" if within(latest[col]) else "🔴 PAGE SOMEONE"
    print(f"  {state:<18} {name:<24} now {latest[col]:<6}  weeks out of SLO: {', '.join(broken) or 'none'}")

In [ ]:
panels = [("auc", "Model · AUC", SLO["auc_min"]), ("predicted_default_rate_pct", "Model · predicted default rate %", None),
          ("top_feature_psi", "Data · top feature PSI", SLO["top_feature_psi_max"]),
          ("p99_latency_ms", "System · p99 latency (ms)", SLO["p99_latency_ms_max"]),
          ("requests_per_hour", "System · requests per hour", None), ("error_rate_pct", "System · error rate %", SLO["error_rate_pct_max"])]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, (col, title, line) in zip(axes.flat, panels):
    ax.plot(data.week, data[col], "o-")
    if line is not None:
        ax.axhline(line, color="crimson", linestyle="--", label="SLO")
        ax.legend()
    ax.set_title(title)
fig.suptitle("Loan default model · weekly MLOps dashboard", fontsize=15, fontweight="bold")
plt.tight_layout()

out = REPO_ROOT / "reports" / "mlops_dashboard.png"
out.parent.mkdir(exist_ok=True)
fig.savefig(out, dpi=120)
plt.show()
print("Saved", out)

Read it as one story: AUC slides toward its SLO while the top feature's PSI climbs into the alert band:
the same drift you measured in Module 3.

## D · The same workflow on Azure Machine Learning (trainer demo)
| Built today | Azure ML equivalent |
|---|---|
| `scripts/main.py` + `config.yaml` | command job on a compute cluster |
| Optuna study | sweep job, with early stopping |
| local MLflow (`mlruns/`) | MLflow tracking in the workspace |
| `model.joblib` | model registry, versioned and tagged |
| FastAPI + Docker | managed online endpoint |
| PSI / KS checks | model monitoring |

## E · Your action plan
Open `docs/WORKSHEET.md` and fill in the five action-plan prompts before you leave.

**Clean up** (Docker): `docker rm -f loan-api`